# 5.3 — Lớp ngữ nghĩa: định nghĩa bằng cấu hình, kiểm chứng bằng dữ liệu

Notebook có 30 cell Markdown/Python, phục vụ Data Analyst và BI. Dùng **Run All** từ đầu; các bảng và nhận xét được lưu ngay trong notebook. Đầu vào là CSV được khai báo trong `config/project.yaml`.

**Một nơi định nghĩa schema:** `config/semantic.yaml` ánh xạ tên cột vật lý sang ID nghiệp vụ, khai báo measure, công thức tỷ lệ, định dạng và quy tắc phân tích. Đổi tên cột CSV thì sửa `source_column` tại cấu hình, không tìm-thay tên trong từng cell. ID nghiệp vụ là hợp đồng ngữ nghĩa; chương trình không tự đoán một cột bất kỳ có phải doanh thu hay không.

**Tự lấy từ dữ liệu:** tên và số nền tảng/quốc gia/ngành, số dòng/cột, khoảng thời gian, nhóm ví dụ, quý tham chiếu và số tổ hợp Platform × Month. Không gán trước kết quả POC để kiểm tra “đúng”.

**Quy tắc có chủ đích:** độ dài kỳ so sánh, ngưỡng số quan sát, các grain Pocket và ngưỡng Spend Level là đầu vào nghiệp vụ có cấu hình, không phải số liệu tự suy ra. Bảng ngưỡng phụ lục được giữ trong `config/spend_thresholds.yaml`; chưa biết thuật toán sinh ngưỡng nên không tự thay bằng quantile.

Grain vẫn là một quan sát nguồn, không khẳng định một chiến dịch duy nhất. Repo chưa có Star Schema vật lý; notebook kiểm chứng logic measure trên CSV đã ép kiểu. Tiền tệ chung là giả định POC có công bố, chưa được nguồn xác nhận. Mặc định không xuất CSV hoặc tạo thư mục đầu ra.

In [31]:
import hashlib
import platform
from decimal import Decimal
from pathlib import Path

import numpy as np
import pandas as pd
import yaml
from IPython.display import Markdown, display

PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / 'config/project.yaml').is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Mở notebook từ thư mục dự án hoặc thư mục con.')
with (PROJECT_ROOT / 'config/project.yaml').open(encoding='utf-8') as file:
    CONFIG = yaml.safe_load(file)
with (PROJECT_ROOT / CONFIG['files']['semantic_config']).open(encoding='utf-8') as file:
    SEM = yaml.safe_load(file)
RAW_PATH = PROJECT_ROOT / CONFIG['paths']['raw_data_dir'] / CONFIG['files']['raw_dataset']
DIMS, MEASURES, RATIOS, DERIVED = (SEM[key] for key in
    ['dimensions', 'measures', 'ratios', 'derived_metrics'])
A, T, SL = (SEM[key] for key in ['analysis', 'technical_columns', 'spend_levels'])
DATE, PRIMARY, SPEND, REVENUE, ROAS, CTR = (A[key] for key in
    ['date_dimension', 'primary_dimension', 'spend_measure', 'revenue_measure', 'roas_metric', 'ctr_metric'])
OBS, OBS_ID, SOURCE_LINE, MONTH, QUARTER = (T[key] for key in
    ['observation_count', 'observation_id', 'source_line', 'month', 'quarter'])
BASE, DIMENSIONS = list(MEASURES), list(DIMS)
COUNT_FIELDS = [key for key, spec in MEASURES.items() if spec['type'] == 'integer']
SOURCE_MAP = {spec['source_column']: key for key, spec in {**DIMS, **MEASURES}.items()}
if len(SOURCE_MAP) != len(DIMS) + len(MEASURES):
    raise ValueError('Nhiều khái niệm đang ánh xạ vào cùng một cột nguồn.')
if set(DIMS) & set(MEASURES) or set(T.values()) & (set(DIMS) | set(MEASURES)):
    raise ValueError('ID nghiệp vụ và cột kỹ thuật bị trùng.')
known = set(MEASURES) | {OBS}
for key, spec in RATIOS.items():
    if spec['numerator'] not in known or spec['denominator'] not in known:
        raise ValueError(f'Tỷ lệ {key} tham chiếu measure không tồn tại.')
known |= set(RATIOS)
for key, spec in DERIVED.items():
    references = ([spec['input']] if spec['operation'] == 'complement'
                  else [spec['numerator'], spec['denominator']])
    if not set(references) <= known:
        raise ValueError(f'Derived metric {key} tham chiếu ID không tồn tại hoặc sai thứ tự.')
    known.add(key)
for key in ['date_dimension', 'primary_dimension']:
    if A[key] not in DIMS:
        raise ValueError(f'Vai trò {key} phải tham chiếu một Dimension.')
if not SEM['source']['assume_common_currency']:
    raise ValueError('Cần bổ sung tiền tệ/quy đổi trước khi tổng hợp xuyên thị trường.')
pd.set_option('display.max_columns', SEM['display']['max_columns'])
pd.set_option('display.max_rows', SEM['display']['max_rows'])
pd.set_option('display.max_colwidth', SEM['display']['max_colwidth'])
EXPORT_PLATFORM_MONTH = SEM['export']['enabled']

def show(title, table, formats=None):
    display(Markdown(f'**{title}**'))
    display(table.style.format(formats, na_rep='—') if formats else table)

def label(key):
    return {**DIMS, **MEASURES, **RATIOS, **DERIVED}.get(key, {}).get('label', key)

display(Markdown(
    f'Python **{platform.python_version()}**, pandas **{pd.__version__}**. '
    f'Đã đọc **{len(DIMS)} Dimension**, **{len(MEASURES)} measure**, '
    f'**{len(RATIOS)} tỷ lệ** từ cấu hình. Tiền tệ chung là giả định, chưa xác nhận USD.'
))

Python **3.12.8**, pandas **2.2.3**. Đã đọc **5 Dimension**, **5 measure**, **5 tỷ lệ** từ cấu hình. Tiền tệ chung là giả định, chưa xác nhận USD.

## 1. Đọc nguồn theo schema đã khai báo

Giữ `raw` nguyên dạng chuỗi. Tạo `df` từ các cột bắt buộc và đổi tên đúng một lần sang ID nghiệp vụ bằng cấu hình. Cột KPI nguồn là tùy chọn để đối chiếu; thiếu chúng vẫn tính KPI từ đại lượng gốc.

Kiểm tra kiểu ngày/số hữu hạn, chiều không rỗng và số đếm nguyên/không âm. Không tự xóa tổ hợp chiều lặp, thay dữ liệu lỗi bằng 0 hoặc gán ngày vắng quan sát thành ngày không hoạt động. Mã quan sát/số dòng chỉ có ý nghĩa trong file đang đọc, không phải campaign_id.

In [32]:
source_hash = hashlib.sha256(RAW_PATH.read_bytes()).hexdigest()
raw = pd.read_csv(RAW_PATH, encoding=SEM['source']['encoding'], dtype=str, keep_default_na=False)
missing_columns = sorted(set(SOURCE_MAP) - set(raw.columns))
if missing_columns or raw.empty:
    raise ValueError(f'Thiếu cột bắt buộc: {missing_columns}; số dòng: {len(raw)}')
df = raw[list(SOURCE_MAP)].rename(columns=SOURCE_MAP).copy(deep=True)
conversion_rows = []
for key, spec in DIMS.items():
    if spec['type'] == 'date':
        parsed = pd.to_datetime(df[key], format=SEM['source']['date_format'], errors='coerce')
        invalid = parsed.isna()
        df[key] = parsed
    else:
        invalid = df[key].str.strip().eq('')
    conversion_rows.append({'semantic_id': key, 'source_column': spec['source_column'],
                            'invalid_values': int(invalid.sum())})
for key, spec in MEASURES.items():
    parsed = pd.to_numeric(df[key], errors='coerce')
    invalid = parsed.isna() | ~np.isfinite(parsed.astype(float)) | parsed.lt(0)
    if spec['type'] == 'integer':
        invalid |= parsed.mod(1).ne(0)
    conversion_rows.append({'semantic_id': key, 'source_column': spec['source_column'],
                            'invalid_values': int(invalid.sum())})
    df[key] = parsed
conversion_audit = pd.DataFrame(conversion_rows)
show('Kiểm tra ánh xạ và chuyển kiểu', conversion_audit)
if conversion_audit.invalid_values.sum():
    raise ValueError('Nguồn không thỏa schema; sửa dữ liệu hoặc cấu hình trước khi tính.')
for key in COUNT_FIELDS:
    df[key] = df[key].astype('int64')
df[OBS_ID] = np.arange(1, len(df) + 1)
df[SOURCE_LINE] = np.arange(2, len(df) + 2)
df[MONTH] = df[DATE].dt.to_period(A['month_frequency']).dt.to_timestamp()
df[QUARTER] = df[DATE].dt.to_period(A['quarter_frequency']).astype(str)
SOURCE_START, SOURCE_END = df[DATE].min(), df[DATE].max()
dimension_counts = df.groupby(DIMENSIONS, dropna=False).size()
missing_days = pd.date_range(SOURCE_START, SOURCE_END).difference(df[DATE].unique())
source_summary = pd.DataFrame([{
    'observations': len(df), 'source_columns': len(raw.columns),
    'date_min': SOURCE_START.date(), 'date_max': SOURCE_END.date(),
    'distinct_dates': df[DATE].nunique(), 'missing_dates_in_range': len(missing_days),
    'unique_dimension_combinations': len(dimension_counts),
    'repeated_dimension_groups': int(dimension_counts.gt(1).sum()),
    'full_duplicate_rows': int(raw.duplicated().sum()),
}])
show('Tóm tắt tự tính từ nguồn', source_summary)
display(Markdown('Ngày vắng quan sát: ' + (', '.join(d.strftime('%Y-%m-%d') for d in missing_days) or 'Không có')))
display(Markdown(f'SHA-256 nguồn: `{source_hash}`'))

**Kiểm tra ánh xạ và chuyển kiểu**

,semantic_id,source_column,invalid_values
0,date,date,0
1,platform,platform,0
2,campaign_type,campaign_type,0
3,industry,industry,0
4,country,country,0
5,impressions,impressions,0
6,clicks,clicks,0
7,ad_spend,ad_spend,0
8,conversions,conversions,0
9,revenue,revenue,0


**Tóm tắt tự tính từ nguồn**

,observations,source_columns,date_min,date_max,distinct_dates,missing_dates_in_range,unique_dimension_combinations,repeated_dimension_groups,full_duplicate_rows
0,1800,14,2024-01-01,2024-12-30,360,5,1788,12,0


Ngày vắng quan sát: 2024-04-23, 2024-07-01, 2024-07-15, 2024-08-25, 2024-11-18

SHA-256 nguồn: `e8cd9ce0ab7f94d473ac3ae220755261f36211e76c3c9b6613b486a23a85838f`

## 2. Từ điển chỉ số được sinh từ cấu hình

Tên nghiệp vụ, nguồn, công thức và đơn vị được lấy cùng nơi với hàm tính. Không giữ một bảng mô tả công thức riêng viết tay có thể lệch khỏi code. ID trong `dimensions`, `measures`, `ratios` là hợp đồng; tên cột vật lý nằm tại `source_column`.

Định nghĩa nghiệp vụ chưa đầy đủ được liệt kê riêng từ `pending_definitions`, không hiển thị như metric đã được tính. Số quan sát không được gọi thành số chiến dịch duy nhất.

In [33]:
catalog_rows = []
for key, spec in MEASURES.items():
    catalog_rows.append({'metric': key, 'business_name': spec['label'], 'kind': 'SUM',
                         'source': spec['source_column'], 'formula': f'SUM({spec["source_column"]})',
                         'unit': spec['unit'], 'format': spec['format']})
catalog_rows.append({'metric': OBS, 'business_name': 'Số quan sát', 'kind': 'COUNT',
                     'source': 'Các dòng hợp lệ', 'formula': 'COUNT_ROWS', 'unit': 'quan sát', 'format': '{:,.0f}'})
for key, spec in RATIOS.items():
    catalog_rows.append({'metric': key, 'business_name': spec['label'], 'kind': 'RATIO_OF_SUMS',
                         'source': spec.get('source_column', 'Không có KPI nguồn'),
                         'formula': f'{spec["numerator"]} / {spec["denominator"]}',
                         'unit': spec['unit'], 'format': spec['format']})
for key, spec in DERIVED.items():
    formula = (f'1 - {spec["input"]}' if spec['operation'] == 'complement'
               else f'{spec["numerator"]} / {spec["denominator"]}')
    catalog_rows.append({'metric': key, 'business_name': spec['label'], 'kind': spec['operation'],
                         'source': 'Measure đã định nghĩa', 'formula': formula,
                         'unit': spec['unit'], 'format': spec['format']})
metric_catalog = pd.DataFrame(catalog_rows)
show('Metric catalog — được dùng chung cho tính toán và hiển thị', metric_catalog)
show('Định nghĩa còn phụ thuộc nhóm/engine DSS', pd.DataFrame(SEM['pending_definitions']))

**Metric catalog — được dùng chung cho tính toán và hiển thị**

,metric,business_name,kind,source,formula,unit,format
0,impressions,Tổng hiển thị,SUM,impressions,SUM(impressions),lượt,"{:,.0f}"
1,clicks,Tổng lượt nhấp,SUM,clicks,SUM(clicks),lượt,"{:,.0f}"
2,ad_spend,Tổng chi phí,SUM,ad_spend,SUM(ad_spend),đơn vị tiền chung giả định,"{:,.2f}"
3,conversions,Tổng chuyển đổi,SUM,conversions,SUM(conversions),chuyển đổi,"{:,.0f}"
4,revenue,Tổng doanh thu,SUM,revenue,SUM(revenue),đơn vị tiền chung giả định,"{:,.2f}"
5,observations,Số quan sát,COUNT,Các dòng hợp lệ,COUNT_ROWS,quan sát,"{:,.0f}"
6,ctr,CTR,RATIO_OF_SUMS,CTR,clicks / impressions,tỷ lệ,{:.2%}
7,cpc,CPC,RATIO_OF_SUMS,CPC,ad_spend / clicks,tiền/lượt nhấp,"{:,.2f}"
8,conversion_rate,Conversion Rate,RATIO_OF_SUMS,Không có KPI nguồn,conversions / clicks,tỷ lệ,{:.2%}
9,cpa,CPA,RATIO_OF_SUMS,CPA,ad_spend / conversions,tiền/chuyển đổi,"{:,.2f}"


**Định nghĩa còn phụ thuộc nhóm/engine DSS**

,metric,reason
0,Growth Opportunity,Grain và tiêu chí gắn nhãn chưa được nhóm chốt
1,Sensitivity và Decision Status,Cần công thức và ngưỡng của engine DSS
2,Simulated Uplift và Win Rate,Cần bảng kịch bản và tập trạng thái đánh giá riêng


## 3. Bộ hàm tổng hợp dùng chung

Hàm tính lặp qua các measure/tỷ lệ trong cấu hình; không chứa danh sách tên cột nguồn. Cộng đại lượng gốc trước, tính tỷ lệ sau. Đổi bộ lọc chỉ đổi tập dòng đầu vào; dòng Tổng phải tính lại từ các đại lượng gốc.

Mẫu số bằng 0 hoặc tập không có quan sát trả về NaN, hiển thị “—”. Không làm tròn dữ liệu trước tính toán. Tỷ lệ được lưu dạng thập phân và chỉ định dạng phần trăm khi hiển thị. Đây là kiểm chứng logic BI, chưa thay thế phép kiểm tra join Dimension–Fact vật lý.

In [34]:
def safe_divide(numerator, denominator):
    num, den = np.broadcast_arrays(np.asarray(numerator, dtype=float), np.asarray(denominator, dtype=float))
    out = np.full(num.shape, np.nan, dtype=float)
    valid = np.isfinite(num) & np.isfinite(den) & (den != 0)
    np.divide(num, den, out=out, where=valid)
    return float(out) if out.ndim == 0 else out

def aggregate_metrics(frame, by=None):
    keys = list(by or [])
    if keys:
        grouped = frame.groupby(keys, dropna=False, observed=True, sort=True)
        totals = grouped[BASE].sum(min_count=1)
        totals[OBS] = grouped.size()
        totals = totals.reset_index()
    else:
        totals = pd.DataFrame([frame[BASE].sum(min_count=1)])
        totals[OBS] = len(frame)
    for key, spec in RATIOS.items():
        totals[key] = safe_divide(totals[spec['numerator']], totals[spec['denominator']])
    for key, spec in DERIVED.items():
        if spec['operation'] == 'complement':
            totals[key] = 1 - totals[spec['input']]
        elif spec['operation'] == 'ratio':
            totals[key] = safe_divide(totals[spec['numerator']], totals[spec['denominator']])
        else:
            raise ValueError(f'Chưa hỗ trợ operation: {spec["operation"]}')
    return totals

def select_scope(frame, start=None, end=None, filters=None):
    mask = pd.Series(True, index=frame.index)
    if start is not None:
        mask &= frame[DATE].ge(pd.Timestamp(start))
    if end is not None:
        mask &= frame[DATE].le(pd.Timestamp(end))
    for key, values in (filters or {}).items():
        if key not in DIMS or key == DATE:
            raise ValueError(f'Chiều không hỗ trợ: {key}')
        accepted = [values] if isinstance(values, str) else list(values)
        mask &= frame[key].isin(accepted)
    return frame.loc[mask].copy()

KPI_FORMATS = {key: spec['format'] for key, spec in {**MEASURES, **RATIOS, **DERIVED}.items()}
KPI_FORMATS[OBS] = '{:,.0f}'
display(Markdown('Đã tạo hàm tính theo schema và công thức trong cấu hình.'))

Đã tạo hàm tính theo schema và công thức trong cấu hình.

## 4. KPI tổng và đối chiếu KPI nguồn tùy chọn

ROAS tổng là tỷ số của tổng doanh thu và tổng chi phí. Trung bình ROAS từng quan sát chỉ mô tả phân phối, không thay thế ROAS tổng. Cả hai được tính từ dữ liệu đang đọc, không so với một hằng số POC.

Các KPI nguồn được đối chiếu theo `source_column`, tử/mẫu và `source_decimals` trong cấu hình. Dung sai bằng nửa đơn vị của chữ số cuối, tính bằng Decimal. Nếu cột KPI nguồn không có hoặc không hợp lệ, báo trạng thái đối chiếu; KPI BI vẫn tính được từ measure cơ sở.

In [35]:
overall = aggregate_metrics(df)
show('Các measure cơ sở', overall[[OBS] + BASE], KPI_FORMATS)
show('KPI và các metric dẫn xuất', overall[list(RATIOS) + list(DERIVED)], KPI_FORMATS)
roas_spec = RATIOS[ROAS]
row_roas = safe_divide(df[roas_spec['numerator']], df[roas_spec['denominator']])
roas_comparison = pd.DataFrame([
    {'definition': f'{label(ROAS)} tổng — tỷ số các tổng', 'value': overall[ROAS].iloc[0]},
    {'definition': f'Trung bình {label(ROAS)} tính lại mỗi quan sát', 'value': pd.Series(row_roas).mean()},
])
optional_roas_column = roas_spec.get('source_column')
if optional_roas_column in raw:
    roas_comparison.loc[len(roas_comparison)] = [
        f'Trung bình KPI nguồn {optional_roas_column}',
        pd.to_numeric(raw[optional_roas_column], errors='coerce').mean(),
    ]
show('Phân biệt cách tổng hợp', roas_comparison, {'value': RATIOS[ROAS]['format']})
audit_rows = []
for key, spec in RATIOS.items():
    source_column = spec.get('source_column')
    if not source_column:
        continue
    if source_column not in raw:
        audit_rows.append({'metric': key, 'source_column': source_column,
                           'status': 'Không có cột đối chiếu tùy chọn', 'compared': 0,
                           'invalid_source_values': np.nan, 'undefined_denominator': np.nan,
                           'outside_tolerance': np.nan, 'tolerance': np.nan, 'max_abs_difference': np.nan})
        continue
    source_values = pd.to_numeric(raw[source_column], errors='coerce')
    valid_source = source_values.notna() & np.isfinite(source_values.astype(float))
    denominator = spec['denominator']
    nonzero = df[denominator].ne(0)
    selected_rows = df.index[valid_source & nonzero]
    num_source = MEASURES[spec['numerator']]['source_column']
    den_source = MEASURES[denominator]['source_column']
    differences = [abs(Decimal(raw.at[i, source_column])
                       - Decimal(raw.at[i, num_source]) / Decimal(raw.at[i, den_source])) for i in selected_rows]
    tolerance = Decimal('0.5').scaleb(-int(spec['source_decimals']))
    audit_rows.append({'metric': key, 'source_column': source_column, 'status': 'Đã đối chiếu',
                       'compared': len(differences), 'invalid_source_values': int((~valid_source).sum()),
                       'undefined_denominator': int((~nonzero).sum()),
                       'outside_tolerance': sum(diff > tolerance for diff in differences),
                       'tolerance': float(tolerance),
                       'max_abs_difference': float(max(differences)) if differences else np.nan})
source_kpi_audit = pd.DataFrame(audit_rows)
show('Kết quả đối chiếu KPI nguồn', source_kpi_audit,
     {'tolerance': '{:.5f}', 'max_abs_difference': '{:.10f}'})
display(Markdown(f'**{label(ROAS)} tổng = {overall[ROAS].iloc[0]:.4f}×**. '
                 'Các chênh lệch nguồn ở bảng trên là phát hiện dữ liệu, không phải kỳ vọng viết sẵn.'))

**Các measure cơ sở**

,observations,impressions,clicks,ad_spend,conversions,revenue
0,"1,800","185,254,234","7,132,816","11,108,749.09","326,812","54,183,330.81"


**KPI và các metric dẫn xuất**

,ctr,cpc,conversion_rate,cpa,roas,no_click_rate,no_conversion_rate,avg_spend_per_observation
0,3.85%,1.56,4.58%,33.99,4.878×,96.15%,95.42%,"6,171.53"


**Phân biệt cách tổng hợp**

,definition,value
0,ROAS tổng — tỷ số các tổng,4.878×
1,Trung bình ROAS tính lại mỗi quan sát,6.450×
2,Trung bình KPI nguồn ROAS,6.450×


**Kết quả đối chiếu KPI nguồn**

,metric,source_column,status,compared,invalid_source_values,undefined_denominator,outside_tolerance,tolerance,max_abs_difference
0,ctr,CTR,Đã đối chiếu,1800,0,0,47,0.00005,0.0001770784
1,cpc,CPC,Đã đối chiếu,1800,0,0,0,0.00500,0.0000000000
2,cpa,CPA,Đã đối chiếu,1800,0,0,0,0.00500,0.0050000000
3,roas,ROAS,Đã đối chiếu,1800,0,0,0,0.00500,0.0049945703


**ROAS tổng = 4.8775×**. Các chênh lệch nguồn ở bảng trên là phát hiện dữ liệu, không phải kỳ vọng viết sẵn.

## 5. Bộ lọc ví dụ được chọn từ các nhóm thực sự có dữ liệu

Tên nền tảng/quốc gia không được gắn sẵn. Notebook chọn tháng hoàn chỉnh gần nhất trong phạm vi nguồn theo cấu hình; nếu không có tháng hoàn chỉnh, dùng tháng quan sát cuối và ghi rõ trạng thái. Trong tháng đó, lấy tổ hợp các chiều ví dụ có nhiều quan sát nhất. Cách chọn này minh họa hành vi lọc, không phải khuyến nghị kinh doanh.

Đổi tên các nhóm trong CSV không làm mất ví dụ. Bảng tổng theo chiều chính và dòng Tổng dùng cùng hàm tính; không trung bình KPI của các nhóm.

In [36]:
platform_metrics = aggregate_metrics(df, [PRIMARY])
platform_with_total = pd.concat([
    platform_metrics, overall.assign(**{PRIMARY: 'TỔNG — tính lại từ nguồn'})
], ignore_index=True)
show(f'Hiệu suất theo {label(PRIMARY)} và Tổng',
     platform_with_total[[PRIMARY, OBS] + BASE + list(RATIOS)], KPI_FORMATS)
example_periods = sorted(df[DATE].dt.to_period(A['month_frequency']).unique())
complete_periods = [p for p in example_periods
                    if p.start_time.normalize() >= SOURCE_START and p.end_time.normalize() <= SOURCE_END]
if A['example_period'] != 'latest_complete_month':
    raise ValueError('Chưa hỗ trợ quy tắc chọn kỳ ví dụ này.')
example_period = complete_periods[-1] if complete_periods else example_periods[-1]
example_month_scope = select_scope(df, example_period.start_time.normalize(), example_period.end_time.normalize())
example_dimensions = A['example_dimensions']
example_combination = (example_month_scope.groupby(example_dimensions, dropna=False)
                       .size().sort_values(ascending=False, kind='stable').index[0])
example_values = example_combination if isinstance(example_combination, tuple) else (example_combination,)
example_filters = dict(zip(example_dimensions, example_values))
example_scope = select_scope(example_month_scope, filters=example_filters)
example_metrics = aggregate_metrics(example_scope)
example_caption = ' × '.join(f'{label(key)}={value}' for key, value in example_filters.items())
show(f'Bộ lọc thực tế: {example_caption}; kỳ {example_period}',
     example_metrics[[OBS] + BASE + list(RATIOS)], KPI_FORMATS)
display(Markdown(f'Ví dụ có **{len(example_scope)} quan sát**; '
                 f'kỳ ví dụ {"được nguồn bao trọn" if complete_periods else "chưa được nguồn bao trọn"}.'))

**Hiệu suất theo Nền tảng và Tổng**

,platform,observations,impressions,clicks,ad_spend,conversions,revenue,ctr,cpc,conversion_rate,cpa,roas
0,Google Ads,720,"73,801,150","2,936,149","6,349,268.91","131,098","22,033,744.95",3.98%,2.16,4.46%,48.43,3.470×
1,Meta Ads,630,"64,403,494","1,596,736","2,106,061.67","73,262","11,926,045.79",2.48%,1.32,4.59%,28.75,5.663×
2,TikTok Ads,450,"47,049,590","2,599,931","2,653,418.51","122,452","20,223,540.07",5.53%,1.02,4.71%,21.67,7.622×
3,TỔNG — tính lại từ nguồn,"1,800","185,254,234","7,132,816","11,108,749.09","326,812","54,183,330.81",3.85%,1.56,4.58%,33.99,4.878×


**Bộ lọc thực tế: Nền tảng=Google Ads × Quốc gia=Canada; kỳ 2024-11**

,observations,impressions,clicks,ad_spend,conversions,revenue,ctr,cpc,conversion_rate,cpa,roas
0,12,"907,177","37,341","81,718.89","1,654","288,178.35",4.12%,2.19,4.43%,49.41,3.526×


Ví dụ có **12 quan sát**; kỳ ví dụ được nguồn bao trọn.

## 6. Tỷ trọng trên phạm vi so sánh

Mẫu số giữ phạm vi slicer nhưng bỏ điều kiện chia nhóm của hàng/cột. Ví dụ tập con các nền tảng được chọn từ giá trị thực tế, với số nhóm minh họa đặt tại cấu hình; không chứa tên cố định trong code.

Matrix dùng hai chiều `cross_dimensions`. Tỷ trọng mỗi ô trên tổng toàn matrix; tổng hàng/cột là cộng các tỷ trọng của các ô. Không lọc Top N trước khi tính mẫu số. Nếu tổng mẫu số bằng 0, tỷ trọng trống và không ép tổng thành 100%.

In [37]:
def contribution_table(scope, by):
    table = aggregate_metrics(scope, by)
    denominator = aggregate_metrics(scope).iloc[0]
    table['revenue_contribution'] = safe_divide(table[REVENUE], denominator[REVENUE])
    table['spend_share'] = safe_divide(table[SPEND], denominator[SPEND])
    return table

SHARE_FORMATS = {**KPI_FORMATS, 'revenue_contribution': '{:.2%}', 'spend_share': '{:.2%}'}
platform_shares = contribution_table(df, [PRIMARY])
show(f'Tỷ trọng theo {label(PRIMARY)}',
     platform_shares[[PRIMARY, REVENUE, SPEND, 'revenue_contribution', 'spend_share']], SHARE_FORMATS)
selected_primary_values = (df[PRIMARY].drop_duplicates().sort_values()
                           .head(A['platform_subset_size']).tolist())
selected_platform_scope = select_scope(df, filters={PRIMARY: selected_primary_values})
selected_platform_shares = contribution_table(selected_platform_scope, [PRIMARY])
show('Tập con nhóm lấy từ dữ liệu — mẫu số giữ tập con này',
     selected_platform_shares[[PRIMARY, 'revenue_contribution', 'spend_share']], SHARE_FORMATS)
cross_dimensions = A['cross_dimensions']
cross_shares = contribution_table(df, cross_dimensions)
contribution_matrix = cross_shares.pivot(index=cross_dimensions[0], columns=cross_dimensions[1],
                                        values='revenue_contribution')
contribution_matrix['Tổng hàng'] = contribution_matrix.sum(axis=1, min_count=1)
contribution_matrix.loc['Tổng cột'] = contribution_matrix.sum(axis=0, min_count=1)
show(' × '.join(label(key) for key in cross_dimensions) + ' — tỷ trọng tổng matrix',
     contribution_matrix, {key: '{:.2%}' for key in contribution_matrix.columns})

**Tỷ trọng theo Nền tảng**

,platform,revenue,ad_spend,revenue_contribution,spend_share
0,Google Ads,"22,033,744.95","6,349,268.91",40.67%,57.16%
1,Meta Ads,"11,926,045.79","2,106,061.67",22.01%,18.96%
2,TikTok Ads,"20,223,540.07","2,653,418.51",37.32%,23.89%


**Tập con nhóm lấy từ dữ liệu — mẫu số giữ tập con này**

,platform,revenue_contribution,spend_share
0,Google Ads,64.88%,75.09%
1,Meta Ads,35.12%,24.91%


**Ngành × Quốc gia — tỷ trọng tổng matrix**

country,Australia,Canada,Germany,India,UAE,UK,USA,Tổng hàng
industry,,,,,,,,
E-commerce,2.27%,4.23%,3.07%,2.39%,1.77%,2.26%,1.79%,17.79%
EdTech,2.89%,2.45%,3.26%,3.59%,3.54%,2.85%,2.74%,21.32%
Fintech,3.29%,1.92%,2.72%,2.99%,2.55%,3.05%,2.26%,18.78%
Healthcare,4.02%,2.27%,2.32%,2.93%,2.76%,2.99%,2.89%,20.17%
SaaS,2.16%,3.64%,2.84%,2.70%,4.03%,2.99%,3.60%,21.95%
Tổng cột,14.62%,14.52%,14.20%,14.60%,14.65%,14.14%,13.27%,100.00%


## 7. Tăng trưởng theo kỳ lịch và khoảng ngày

Các measure Growth được khai báo ở cấu hình. Với mỗi measure: (hiện tại − trước) / trước. Kỳ lịch được tạo từ min–max ngày nguồn; reindex đầy đủ để không nhảy qua kỳ trống. Quý/tháng cuối chưa được nguồn bao trọn sẽ không có Growth theo quy ước thận trọng đề xuất; tổng quan sát vẫn hiện.

Khoảng so sánh lấy ngày cuối nguồn và số ngày từ cấu hình. Kỳ trước có cùng số ngày lịch, đứng ngay trước kỳ chọn; giữ bộ lọc nghiệp vụ. Không có đủ kỳ trước thì hiện trống. Notebook không chứa ngày tháng năm cụ thể. Matrix kỳ lịch không dùng bộ lọc ngày của card.

In [38]:
def calendar_growth(frame, frequency):
    work = frame.copy()
    work['period'] = work[DATE].dt.to_period(frequency)
    table = aggregate_metrics(work, ['period']).set_index('period')
    table = table.reindex(pd.period_range(SOURCE_START, SOURCE_END, freq=frequency))
    table.index.name = 'period'
    table['range_covers_period'] = [p.start_time.normalize() >= SOURCE_START
                                   and p.end_time.normalize() <= SOURCE_END for p in table.index]
    previous_complete = table.range_covers_period.shift(1, fill_value=False)
    for measure in A['growth_measures']:
        previous = table[measure].shift(1)
        growth = f'{measure}_growth'
        table[growth] = safe_divide(table[measure] - previous, previous)
        table.loc[~(table.range_covers_period & previous_complete), growth] = np.nan
    return table.reset_index()

def compare_window(scope, start, end):
    start, end = pd.Timestamp(start), pd.Timestamp(end)
    if end < start:
        raise ValueError('Ngày kết thúc phải từ ngày bắt đầu trở đi.')
    length = (end - start).days + 1
    previous_start, previous_end = start - pd.Timedelta(days=length), start - pd.Timedelta(days=1)
    current = aggregate_metrics(select_scope(scope, start, end)).iloc[0]
    previous = aggregate_metrics(select_scope(scope, previous_start, previous_end)).iloc[0]
    current_supported = SOURCE_START <= start <= end <= SOURCE_END
    previous_supported = SOURCE_START <= previous_start <= previous_end <= SOURCE_END
    usable = current_supported and previous_supported
    status = ('Có kỳ so sánh trong nguồn' if usable else 'Không có kỳ trước'
              if not previous_supported else 'Kỳ chọn vượt phạm vi nguồn')
    if usable and (current[OBS] == 0 or previous[OBS] == 0):
        status = 'Không có quan sát ở kỳ chọn/kỳ trước trong bộ lọc'
    return pd.DataFrame([{
        'measure': measure, 'current': current[measure], 'previous': previous[measure],
        'growth': safe_divide(current[measure] - previous[measure], previous[measure]) if usable else np.nan,
        'current_start': start.date(), 'current_end': end.date(),
        'previous_start': previous_start.date(), 'previous_end': previous_end.date(),
        'days_each_period': length, 'status': status,
    } for measure in A['growth_measures']])

monthly_growth = calendar_growth(df, A['month_frequency'])
quarterly_growth = calendar_growth(df, A['quarter_frequency'])
growth_columns = ['period', OBS] + A['growth_measures'] + [ROAS] + [
    f'{measure}_growth' for measure in A['growth_measures']] + ['range_covers_period']
growth_formats = {**KPI_FORMATS, **{f'{measure}_growth': '{:+.2%}' for measure in A['growth_measures']}}
show('Growth theo tháng lịch', monthly_growth[growth_columns], growth_formats)
show('Growth theo quý lịch', quarterly_growth[growth_columns], growth_formats)
window_end = SOURCE_END
window_days = int(A['comparison_window_days'])
window_start = window_end - pd.Timedelta(days=window_days - 1)
window_comparison = compare_window(df, window_start, window_end)
show(f'{window_days} ngày cuối nguồn so với khoảng liền trước', window_comparison,
     {'current': '{:,.2f}', 'previous': '{:,.2f}', 'growth': '{:+.2%}'})

**Growth theo tháng lịch**

,period,observations,ad_spend,revenue,roas,ad_spend_growth,revenue_growth,range_covers_period
0,2024-01,166,"901,735.06","4,691,994.47",5.203×,—,—,True
1,2024-02,156,"988,588.36","4,749,203.46",4.804×,+9.63%,+1.22%,True
2,2024-03,147,"837,915.28","4,407,519.38",5.260×,-15.24%,-7.19%,True
3,2024-04,164,"1,032,895.90","5,143,463.57",4.980×,+23.27%,+16.70%,True
4,2024-05,144,"904,564.26","3,768,913.36",4.167×,-12.42%,-26.72%,True
5,2024-06,152,"966,952.76","4,362,176.31",4.511×,+6.90%,+15.74%,True
6,2024-07,126,"797,918.14","3,695,093.97",4.631×,-17.48%,-15.29%,True
7,2024-08,152,"819,622.05","4,301,659.26",5.248×,+2.72%,+16.42%,True
8,2024-09,161,"1,010,523.04","4,749,430.27",4.700×,+23.29%,+10.41%,True
9,2024-10,132,"868,099.26","4,670,520.68",5.380×,-14.09%,-1.66%,True


**Growth theo quý lịch**

,period,observations,ad_spend,revenue,roas,ad_spend_growth,revenue_growth,range_covers_period
0,2024Q1,469,"2,728,238.70","13,848,717.31",5.076×,—,—,True
1,2024Q2,460,"2,904,412.92","13,274,553.24",4.570×,+6.46%,-4.15%,True
2,2024Q3,439,"2,628,063.23","12,746,183.50",4.850×,-9.51%,-3.98%,True
3,2024Q4,432,"2,848,034.24","14,313,876.76",5.026×,—,—,False


**90 ngày cuối nguồn so với khoảng liền trước**

,measure,current,previous,growth,current_start,current_end,previous_start,previous_end,days_each_period,status
0,ad_spend,"2,842,636.40","2,581,127.18",+10.13%,2024-10-02,2024-12-30,2024-07-04,2024-10-01,90,Có kỳ so sánh trong nguồn
1,revenue,"14,273,713.83","12,675,690.20",+12.61%,2024-10-02,2024-12-30,2024-07-04,2024-10-01,90,Có kỳ so sánh trong nguồn


## 8. Phân phối và biên độ hiệu suất

Chọn metric ROAS và danh sách các chiều so sánh từ cấu hình. Phân phối dùng tỷ lệ từng quan sát, biên độ dùng tỷ lệ của tổng trong từng nhóm. Độ rộng và ngưỡng nhóm cuối có cấu hình; không viết sẵn danh sách nhãn.

Các khoảng thông thường đóng trái/mở phải; khoảng ngay trước ngưỡng cuối đóng cả phải, nhóm cuối lớn hơn ngưỡng. Giá trị chưa xác định được đếm riêng. Chênh lệch hiệu suất chỉ là mô tả, chưa phải bằng chứng nhân quả.

In [39]:
roas_spec = RATIOS[ROAS]
row_roas = safe_divide(df[roas_spec['numerator']], df[roas_spec['denominator']])
histogram = A['roas_histogram']
width, overflow = float(histogram['width']), float(histogram['overflow'])
if width <= 0 or overflow <= width or not np.isclose(overflow / width, round(overflow / width)):
    raise ValueError('Histogram yêu cầu ngưỡng dương và chia hết cho độ rộng.')
edges = np.arange(0, overflow, width)
conditions, band_labels = [], []
for index, left in enumerate(edges):
    right = left + width
    last = index == len(edges) - 1
    conditions.append((row_roas >= left) & ((row_roas <= right) if last else (row_roas < right)))
    band_labels.append(f'[{left:g},{right:g}{"]" if last else ")"}')
conditions.append(row_roas > overflow)
band_labels.append(f'>{overflow:g}')
undefined_band = 'Không xác định'
banded = pd.Categorical(np.select(conditions, band_labels, default=undefined_band),
                       categories=band_labels + [undefined_band], ordered=True)
roas_distribution = pd.Series(banded).value_counts(sort=False).rename(OBS).rename_axis('band').reset_index()
roas_distribution['share'] = safe_divide(roas_distribution[OBS], len(df))
show('Phân phối hiệu suất quan sát', roas_distribution, {OBS: '{:,.0f}', 'share': '{:.2%}'})
range_rows = []
for dimension in A['range_dimensions']:
    grouped = aggregate_metrics(df, [dimension]).dropna(subset=[ROAS])
    if grouped.empty:
        range_rows.append({'dimension': dimension, 'groups': 0, 'roas_range': np.nan})
        continue
    low, high = grouped.loc[grouped[ROAS].idxmin()], grouped.loc[grouped[ROAS].idxmax()]
    range_rows.append({'dimension': dimension, 'groups': len(grouped),
                       'lowest_group': low[dimension], 'lowest_roas': low[ROAS], 'low_observations': low[OBS],
                       'highest_group': high[dimension], 'highest_roas': high[ROAS], 'high_observations': high[OBS],
                       'roas_range': high[ROAS] - low[ROAS]})
roas_ranges = pd.DataFrame(range_rows).sort_values('roas_range', ascending=False)
show('Biên độ hiệu suất nhóm theo các chiều đã cấu hình', roas_ranges,
     {key: RATIOS[ROAS]['format'] for key in ['lowest_roas', 'highest_roas', 'roas_range']})

**Phân phối hiệu suất quan sát**

,band,observations,share
0,"[0,2)",410,22.78%
1,"[2,4)",438,24.33%
2,"[4,6)",289,16.06%
3,"[6,8)",194,10.78%
4,"[8,10)",125,6.94%
5,"[10,12)",99,5.50%
6,"[12,14)",60,3.33%
7,"[14,16)",43,2.39%
8,"[16,18)",28,1.56%
9,"[18,20]",29,1.61%


**Biên độ hiệu suất nhóm theo các chiều đã cấu hình**

,dimension,groups,lowest_group,lowest_roas,low_observations,highest_group,highest_roas,high_observations,roas_range
0,platform,3,Google Ads,3.470×,720,TikTok Ads,7.622×,450,4.151×
3,country,7,USA,4.162×,266,India,5.498×,261,1.335×
1,campaign_type,4,Shopping,4.581×,447,Search,5.306×,477,0.725×
2,industry,5,Fintech,4.479×,361,SaaS,5.044×,370,0.566×


## 9. Phân loại chi tiêu bằng chính sách đầu vào

Đọc bảng ngưỡng từ đường dẫn cấu hình; tên chiều nối bảng, tên cột ngưỡng và nhãn mức đều được cấu hình. Không chứa danh sách quốc gia/ngành hoặc số lượng bộ ngưỡng trong code. Các giá trị ngưỡng của phụ lục là chính sách được lưu có nguồn gốc, không được thay bằng ngưỡng tự đoán từ dữ liệu.

Nhóm mới chưa có chính sách được giữ nhãn Unclassified theo cấu hình, có báo số lượng và giữ nguyên tổng; không tự gán High, không xóa dòng. Tất cả các ngưỡng phải có khóa duy nhất và biên hợp lệ. Khi thay tên giá trị Country/Industry, cập nhật bảng chính sách tương ứng hoặc xem các nhóm Unclassified để bổ sung.

In [40]:
with (PROJECT_ROOT / SL['thresholds_file']).open(encoding='utf-8') as file:
    threshold_config = yaml.safe_load(file)
spend_thresholds = pd.DataFrame(threshold_config['records'])
THRESHOLD_DIMS = SL['dimensions']
LOW, HIGH, LEVEL = (SL[key] for key in ['low_bound_column', 'high_bound_column', 'column'])
SPEND_ORDER, UNCLASSIFIED = SL['labels'], SL['unknown_label']
if len(SPEND_ORDER) != 3 or len(set(SPEND_ORDER + [UNCLASSIFIED])) != 4:
    raise ValueError('Chính sách hai ngưỡng cần ba nhãn mức khác nhau và một nhãn chưa phân loại.')
if not set(THRESHOLD_DIMS + [LOW, HIGH]) <= set(spend_thresholds):
    raise ValueError('Bảng ngưỡng không thỏa schema đã cấu hình.')
if spend_thresholds.duplicated(THRESHOLD_DIMS).any():
    raise ValueError('Khóa bảng ngưỡng không duy nhất.')
for key in [LOW, HIGH]:
    spend_thresholds[key] = pd.to_numeric(spend_thresholds[key], errors='raise')
if not (np.isfinite(spend_thresholds[[LOW, HIGH]].to_numpy()).all()
        and spend_thresholds[LOW].ge(0).all() and spend_thresholds[LOW].lt(spend_thresholds[HIGH]).all()):
    raise ValueError('Bảng ngưỡng có biên không hợp lệ.')
df = df.merge(spend_thresholds[THRESHOLD_DIMS + [LOW, HIGH]], on=THRESHOLD_DIMS,
              how='left', validate='many_to_one', sort=False)

def classify_spend(values, low, high):
    values, low, high = np.broadcast_arrays(np.asarray(values, dtype=float),
                                          np.asarray(low, dtype=float), np.asarray(high, dtype=float))
    valid = np.isfinite(values) & np.isfinite(low) & np.isfinite(high) & (values >= 0) & (low < high)
    return np.select([valid & (values <= low), valid & (values > low) & (values <= high),
                      valid & (values > high)], SPEND_ORDER, default=UNCLASSIFIED)

classified = classify_spend(df[SPEND], df[LOW], df[HIGH])
if (classified == UNCLASSIFIED).any() and SL['missing_policy'] != 'keep_unclassified':
    raise ValueError('Thiếu chính sách phân loại; chưa hỗ trợ missing_policy này.')
df[LEVEL] = pd.Categorical(classified, categories=SPEND_ORDER + [UNCLASSIFIED], ordered=True)
show('Bảng ngưỡng đầu vào — xem trước', spend_thresholds.head(SEM['display']['preview_rows']),
     {LOW: '{:,.2f}', HIGH: '{:,.2f}'})
spend_counts = df.groupby(LEVEL, observed=False).size().rename(OBS).reset_index()
show('Số quan sát mỗi mức, bao gồm nhóm chưa có chính sách', spend_counts)
missing_policy_groups = df.loc[df[LEVEL].eq(UNCLASSIFIED), THRESHOLD_DIMS].drop_duplicates()
display(Markdown(f'Đã đọc **{len(spend_thresholds)} bộ ngưỡng**; '
                 f'**{int(df[LEVEL].eq(UNCLASSIFIED).sum())} quan sát chưa phân loại** '
                 f'thuộc **{len(missing_policy_groups)} tổ hợp chưa có chính sách**.'))
if not missing_policy_groups.empty:
    show('Các tổ hợp cần bổ sung chính sách', missing_policy_groups)

**Bảng ngưỡng đầu vào — xem trước**

,country,industry,low_max,medium_max
0,UAE,Fintech,"5,239.86","13,582.33"
1,UAE,EdTech,"3,823.26","10,231.12"
2,UAE,Healthcare,"5,706.85","14,074.34"
3,UAE,SaaS,"4,595.14","13,020.79"
4,UAE,E-commerce,"4,200.44","8,729.09"
5,UK,Fintech,"5,536.50","12,606.41"


**Số quan sát mỗi mức, bao gồm nhóm chưa có chính sách**

,spend_level,observations
0,Low,1107
1,Medium,499
2,High,194
3,Unclassified,0


Đã đọc **35 bộ ngưỡng**; **0 quan sát chưa phân loại** thuộc **0 tổ hợp chưa có chính sách**.

## 10. Response theo thứ tự mức chi tiêu đã cấu hình

ROAS/CPA theo mức chi tiêu dùng lại measure chung. Với mỗi outcome ở `response_measures`, tính chênh lệch tổng outcome chia chênh lệch tổng chi phí giữa hai mức kề. Không nhảy qua mức vắng hoặc so sánh giữa các nhóm khác nhau.

Mức đầu và Unclassified không có Response; thiếu mức trước/ΔSpend bằng 0 trả về trống. ΔSpend âm có cờ riêng và không được diễn giải thành tăng ngân sách. Luôn đọc kèm số quan sát, do quy mô và cơ cấu nhóm có thể khác nhau. Đây là so sánh mô tả, chưa phải tác động cận biên.

In [41]:
RESPONSE_COLS = {measure: f'{measure}_response' for measure in A['response_measures']}

def spend_response(frame, dimensions=None):
    dimensions = list(dimensions or [])
    levels = SPEND_ORDER + ([UNCLASSIFIED] if frame[LEVEL].eq(UNCLASSIFIED).any() else [])
    scaffold = pd.DataFrame({LEVEL: levels})
    if dimensions:
        scaffold = frame[dimensions].drop_duplicates().merge(scaffold, how='cross')
    table = scaffold.merge(aggregate_metrics(frame, dimensions + [LEVEL]),
                           on=dimensions + [LEVEL], how='left', validate='one_to_one')
    table['level_order'] = table[LEVEL].map({value: i for i, value in enumerate(levels)})
    table = table.sort_values(dimensions + ['level_order']).reset_index(drop=True)
    previous = {}
    for key in list(dict.fromkeys([SPEND, OBS] + A['response_measures'])):
        previous[key] = (table.groupby(dimensions, observed=True, dropna=False)[key].shift(1)
                         if dimensions else table[key].shift(1))
    table['delta_spend'] = table[SPEND] - previous[SPEND]
    valid_pair = table[OBS].gt(0) & previous[OBS].gt(0) & table[LEVEL].isin(SPEND_ORDER)
    for measure, output in RESPONSE_COLS.items():
        table[output] = safe_divide(table[measure] - previous[measure], table.delta_spend)
        table.loc[~valid_pair, output] = np.nan
    table['comparison_status'] = np.select(
        [table[LEVEL].eq(UNCLASSIFIED), ~valid_pair, table.delta_spend.eq(0), table.delta_spend.lt(0)],
        ['Chưa có chính sách mức chi tiêu', 'Thiếu mức liền trước/quan sát',
         'ΔSpend = 0', 'ΔSpend âm: không gọi là tăng chi tiêu'], default='So sánh mô tả, ΔSpend dương')
    return table

spend_overall = spend_response(df)
spend_by_platform = spend_response(df, [PRIMARY])
response_columns = [LEVEL, OBS] + list(dict.fromkeys([SPEND] + A['response_measures']))
response_columns += list(RATIOS) + ['delta_spend'] + list(RESPONSE_COLS.values()) + ['comparison_status']
response_formats = {**KPI_FORMATS, 'delta_spend': '{:+,.2f}',
                    **{key: '{:.6f}' for key in RESPONSE_COLS.values()}}
show('Response toàn phạm vi', spend_overall[response_columns], response_formats)
show(f'Response theo {label(PRIMARY)}', spend_by_platform[[PRIMARY] + response_columns], response_formats)

**Response toàn phạm vi**

,spend_level,observations,ad_spend,revenue,conversions,ctr,cpc,conversion_rate,cpa,roas,delta_spend,revenue_response,conversions_response,comparison_status
0,Low,"1,107","3,002,315.65","21,653,167.60","130,838",3.14%,1.07,4.67%,22.95,7.212×,—,—,—,Thiếu mức liền trước/quan sát
1,Medium,499,"4,549,881.24","20,491,510.56","125,142",4.22%,1.63,4.48%,36.36,4.504×,"+1,547,565.59",-0.750635,-0.003681,"So sánh mô tả, ΔSpend dương"
2,High,194,"3,556,552.20","12,038,652.65","70,832",5.16%,2.31,4.60%,50.21,3.385×,"-993,329.04",8.509625,0.054675,ΔSpend âm: không gọi là tăng chi tiêu


**Response theo Nền tảng**

,platform,spend_level,observations,ad_spend,revenue,conversions,ctr,cpc,conversion_rate,cpa,roas,delta_spend,revenue_response,conversions_response,comparison_status
0,Google Ads,Low,299,"943,613.43","4,644,267.47","28,851",3.28%,1.55,4.75%,32.71,4.922×,—,—,—,Thiếu mức liền trước/quan sát
1,Google Ads,Medium,270,"2,550,077.00","9,238,366.82","55,132",3.85%,2.02,4.37%,46.25,3.623×,"+1,606,463.57",2.859759,0.016360,"So sánh mô tả, ΔSpend dương"
2,Google Ads,High,151,"2,855,578.48","8,151,110.66","47,115",4.74%,2.68,4.42%,60.61,2.854×,"+305,501.48",-3.558923,-0.026242,"So sánh mô tả, ΔSpend dương"
3,Meta Ads,Low,527,"1,231,636.30","8,329,449.66","51,093",2.28%,1.12,4.63%,24.11,6.763×,—,—,—,Thiếu mức liền trước/quan sát
4,Meta Ads,Medium,99,"816,740.98","3,393,662.04","20,696",3.02%,1.75,4.43%,39.46,4.155×,"-414,895.32",11.896465,0.073264,ΔSpend âm: không gọi là tăng chi tiêu
5,Meta Ads,High,4,"57,684.39","202,934.09","1,473",4.08%,2.10,5.36%,39.16,3.518×,"-759,056.59",4.203544,0.025325,ΔSpend âm: không gọi là tăng chi tiêu
6,TikTok Ads,Low,281,"827,065.92","8,679,450.47","50,894",4.87%,0.76,4.67%,16.25,10.494×,—,—,—,Thiếu mức liền trước/quan sát
7,TikTok Ads,Medium,130,"1,183,063.26","7,859,481.70","49,314",5.92%,1.11,4.63%,23.99,6.643×,"+355,997.34",-2.303300,-0.004438,"So sánh mô tả, ΔSpend dương"
8,TikTok Ads,High,39,"643,289.33","3,684,607.90","22,244",6.67%,1.45,5.00%,28.92,5.728×,"-539,773.93",7.734486,0.050151,ΔSpend âm: không gọi là tăng chi tiêu


## 11. Growth Pocket: grain và điều kiện hỗ trợ có cấu hình

Các grain lấy từ `pocket_grains`; không viết lại danh sách chiều trong hàm tính. Ngưỡng quan sát và số dòng xem trước lấy từ cấu hình. Đếm số nhóm quan sát thực tế, không giả định mọi tổ hợp Cartesian đều xuất hiện.

Điều kiện đủ số quan sát chỉ phản ánh hỗ trợ dữ liệu, chưa đủ để gắn nhãn Growth Opportunity. Bảng nhóm doanh thu lớn nhất là bảng mô tả, không phải danh sách khuyến nghị. Mẫu số tỷ trọng lấy trước khi cắt phần xem trước.

In [42]:
pocket_tables, coverage_rows = {}, []
minimum_observations = int(A['pocket_min_observations'])
for spec in A['pocket_grains']:
    pockets = contribution_table(df, spec['dimensions'])
    pockets['sufficient_observations'] = pockets[OBS].ge(minimum_observations)
    pocket_tables[spec['label']] = pockets
    supported = pockets.loc[pockets.sufficient_observations]
    coverage_rows.append({'grain': spec['label'], 'observed_groups': len(pockets),
                          'min_observations': pockets[OBS].min(),
                          'median_observations': pockets[OBS].median(),
                          'max_observations': pockets[OBS].max(),
                          'minimum_required': minimum_observations, 'supported_groups': len(supported),
                          'supported_observations': supported[OBS].sum(),
                          'observation_coverage': safe_divide(supported[OBS].sum(), len(df))})
pocket_coverage = pd.DataFrame(coverage_rows)
show('Độ bao phủ theo grain và ngưỡng trong cấu hình', pocket_coverage,
     {'observation_coverage': '{:.2%}', 'median_observations': '{:.1f}'})
first_pocket_spec = A['pocket_grains'][0]
top_rows = int(SEM['display']['top_rows'])
pocket_example = pocket_tables[first_pocket_spec['label']].sort_values(REVENUE, ascending=False).head(top_rows)
show(f'Xem trước tối đa {top_rows} nhóm doanh thu lớn — chưa gắn nhãn cơ hội',
     pocket_example[first_pocket_spec['dimensions'] + [OBS] + BASE + list(RATIOS)
                    + ['revenue_contribution', 'spend_share', 'sufficient_observations']], SHARE_FORMATS)

**Độ bao phủ theo grain và ngưỡng trong cấu hình**

,grain,observed_groups,min_observations,median_observations,max_observations,minimum_required,supported_groups,supported_observations,observation_coverage
0,Platform × Country × Campaign Type,84,10,20.5,36,10,84,1800,100.00%
1,Platform × Campaign Type × Industry × Country,407,1,4.0,13,10,11,118,6.56%


**Xem trước tối đa 10 nhóm doanh thu lớn — chưa gắn nhãn cơ hội**

,platform,country,campaign_type,observations,impressions,clicks,ad_spend,conversions,revenue,ctr,cpc,conversion_rate,cpa,roas,revenue_contribution,spend_share,sufficient_observations
17,Google Ads,UAE,Search,35,"3,747,809","147,985","278,088.24","6,504","1,284,661.98",3.95%,1.88,4.40%,42.76,4.620×,2.37%,2.50%,True
57,TikTok Ads,Australia,Search,16,"1,952,897","109,675","71,559.51","6,004","1,220,030.53",5.62%,0.65,5.47%,11.92,17.049×,2.25%,0.64%,True
9,Google Ads,Germany,Search,30,"3,345,352","133,559","275,484.09","6,611","1,166,961.86",3.99%,2.06,4.95%,41.67,4.236×,2.15%,2.48%,True
61,TikTok Ads,Canada,Search,23,"2,195,470","131,194","137,529.04","6,947","1,090,636.34",5.98%,1.05,5.30%,19.80,7.930×,2.01%,1.24%,True
74,TikTok Ads,UAE,Shopping,22,"2,504,108","149,933","152,609.10","7,780","1,090,236.67",5.99%,1.02,5.19%,19.62,7.144×,2.01%,1.37%,True
76,TikTok Ads,UK,Display,20,"2,148,523","102,603","105,857.96","5,337","1,063,584.17",4.78%,1.03,5.20%,19.83,10.047×,1.96%,0.95%,True
25,Google Ads,USA,Search,30,"2,858,508","115,652","252,008.52","6,652","1,049,001.65",4.05%,2.18,5.75%,37.88,4.163×,1.94%,2.27%,True
0,Google Ads,Australia,Display,33,"3,259,573","124,502","307,947.55","6,009","1,025,370.09",3.82%,2.47,4.83%,51.25,3.330×,1.89%,2.77%,True
23,Google Ads,UK,Video,26,"2,888,926","124,830","232,316.15","5,756","1,017,791.51",4.32%,1.86,4.61%,40.36,4.381×,1.88%,2.09%,True
70,TikTok Ads,India,Shopping,18,"2,357,856","134,877","135,156.41","5,609","1,013,180.00",5.72%,1.00,4.16%,24.10,7.496×,1.87%,1.22%,True


## 12. Đầu vào DSS và Baseline lấy từ kỳ cuối của nguồn

Tổng hợp theo tháng × chiều nền tảng đã cấu hình. Các hiệu suất tính trên outcome được khai báo trong `yield_measures`. Số nhóm thực tế và tình trạng thiếu tổ hợp được suy ra từ dữ liệu, không mặc định có một số dòng cố định.

Baseline sử dụng **quý chứa ngày cuối nguồn**, theo quy tắc `latest_observed_quarter`. Với dữ liệu ban đầu, quy tắc này khớp Q4 trong tài liệu POC; khi đổi năm dữ liệu thì không lấy lại quý của năm cũ. Số tháng mẫu số tính từ biên quý lịch; nếu thiếu hẳn tháng thì không tính ngân sách tháng tham chiếu. Nếu có đủ tháng nhưng nguồn chưa tới ngày cuối quý, giữ kết quả có công bố giới hạn phạm vi.

Không so kết quả mới với tỷ trọng/ngân sách POC cố định. Không chạy Candidate/Maximin, không dùng Baseline toàn mẫu cho fold quá khứ. Các kết quả mô phỏng/sensitivity vẫn cần mô hình và bảng kịch bản riêng.

In [43]:
platform_month = aggregate_metrics(df, [MONTH, PRIMARY])
YIELD_COLS = {measure: f'{measure}_per_unit_spend' for measure in A['yield_measures']}
for measure, output in YIELD_COLS.items():
    platform_month[output] = safe_divide(platform_month[measure], platform_month[SPEND])
pm_columns = [MONTH, PRIMARY, OBS] + BASE + list(YIELD_COLS.values())
expected_months = pd.date_range(SOURCE_START.to_period(A['month_frequency']).start_time,
                                SOURCE_END.to_period(A['month_frequency']).start_time, freq='MS')
expected_primary_values = df[PRIMARY].drop_duplicates().sort_values()
expected_platform_month = pd.MultiIndex.from_product([expected_months, expected_primary_values], names=[MONTH, PRIMARY])
observed_platform_month = pd.MultiIndex.from_frame(platform_month[[MONTH, PRIMARY]])
missing_platform_month = expected_platform_month.difference(observed_platform_month)
preview_rows = int(SEM['display']['preview_rows'])
show(f'Đầu vào DSS — {min(preview_rows, len(platform_month))}/{len(platform_month)} dòng',
     platform_month[pm_columns].head(preview_rows),
     {**KPI_FORMATS, **{key: '{:.6f}' for key in YIELD_COLS.values()}})
if A['baseline_period'] != 'latest_observed_quarter':
    raise ValueError('Chưa hỗ trợ quy tắc Baseline này.')
baseline_period = SOURCE_END.to_period(A['quarter_frequency'])
baseline_start, baseline_end = baseline_period.start_time.normalize(), baseline_period.end_time.normalize()
baseline_months = pd.period_range(baseline_start, baseline_end, freq=A['month_frequency'])
baseline_scope = select_scope(df, baseline_start, baseline_end)
observed_baseline_months = baseline_scope[DATE].dt.to_period(A['month_frequency']).nunique()
baseline_has_all_months = observed_baseline_months == len(baseline_months)
baseline_monthly_budget = (baseline_scope[SPEND].sum() / len(baseline_months)
                           if baseline_has_all_months else np.nan)
baseline_allocation = contribution_table(baseline_scope, [PRIMARY])[[PRIMARY, SPEND, 'spend_share']]
baseline_allocation['monthly_baseline_budget'] = baseline_monthly_budget * baseline_allocation.spend_share
show(f'Baseline {baseline_period} — tính từ nguồn hiện tại', baseline_allocation,
     {**SHARE_FORMATS, 'monthly_baseline_budget': '{:,.2f}'})
display(Markdown(
    f'Có **{len(platform_month)} tổ hợp quan sát**, '
    f'**{len(missing_platform_month)} tổ hợp vắng** trong lưới tháng × nhóm thực tế. '
    f'Baseline có **{observed_baseline_months}/{len(baseline_months)} tháng**, '
    f'ngân sách tham chiếu **{baseline_monthly_budget:,.2f} đơn vị tiền/tháng**. '
    f'Phạm vi nguồn {"bao trọn" if SOURCE_END >= baseline_end else "chưa bao trọn"} quý tham chiếu.'
))

**Đầu vào DSS — 6/36 dòng**

,month,platform,observations,impressions,clicks,ad_spend,conversions,revenue,revenue_per_unit_spend,conversions_per_unit_spend,impressions_per_unit_spend
0,2024-01-01 00:00:00,Google Ads,61,"5,374,750","195,369","417,210.55","8,970","1,440,502.10",3.452698,0.021500,12.882584
1,2024-01-01 00:00:00,Meta Ads,64,"7,463,528","186,457","229,950.39","8,641","1,569,120.08",6.823733,0.037578,32.457123
2,2024-01-01 00:00:00,TikTok Ads,41,"4,137,760","230,249","254,574.12","10,434","1,682,372.29",6.608575,0.040986,16.253655
3,2024-02-01 00:00:00,Google Ads,69,"6,348,968","259,429","544,783.84","11,146","1,879,131.58",3.449316,0.020459,11.654105
4,2024-02-01 00:00:00,Meta Ads,45,"4,625,254","123,078","187,842.09","5,624","920,932.37",4.902694,0.029940,24.623097
5,2024-02-01 00:00:00,TikTok Ads,42,"4,416,464","255,369","255,962.43","11,793","1,949,139.51",7.614944,0.046073,17.254345


**Baseline 2024Q4 — tính từ nguồn hiện tại**

,platform,ad_spend,spend_share,monthly_baseline_budget
0,Google Ads,"1,576,855.79",55.37%,"525,618.60"
1,Meta Ads,"520,177.92",18.26%,"173,392.64"
2,TikTok Ads,"751,000.53",26.37%,"250,333.51"


Có **36 tổ hợp quan sát**, **0 tổ hợp vắng** trong lưới tháng × nhóm thực tế. Baseline có **3/3 tháng**, ngân sách tham chiếu **949,344.75 đơn vị tiền/tháng**. Phạm vi nguồn chưa bao trọn quý tham chiếu.

## 13. Kiểm tra thuộc tính bất biến thay cho kết quả POC viết sẵn

Kiểm tra bảo toàn số dòng/tổng, tính lại tỷ lệ, tính tỷ trọng theo mẫu số, duy nhất khóa nhóm, kỳ so sánh và xử lý trường hợp biên. Không kiểm tra “phải có đúng một số cột/ngưỡng/tổ hợp cố định” hoặc “ngân sách phải bằng POC”. Số nhóm hợp lệ được đối chiếu với chính các nhóm quan sát, không với hằng số.

Ngưỡng chính sách chưa bao phủ nhóm mới là cảnh báo dữ liệu riêng; nhóm vẫn được giữ trong Unclassified. Các phép thử biên lấy giá trị ngưỡng từ cấu hình, thời điểm thử từ dữ liệu; có trạng thái SKIP nếu nguồn quá ngắn để thực hiện một phép kiểm tra cần nhiều kỳ.

In [44]:
checks = []
RTOL, ATOL = SEM['validation']['rtol'], SEM['validation']['atol']

def check(name, condition, detail, skip=False):
    checks.append({'check': name, 'result': 'SKIP' if skip else 'PASS' if bool(condition) else 'FAIL', 'detail': detail})

def close(left, right):
    return np.allclose(left, right, rtol=RTOL, atol=ATOL, equal_nan=True)

check('Bảo toàn số quan sát và mã dòng', len(df) == len(raw) and df[OBS_ID].is_unique,
      'Không loại tổ hợp chiều lặp, không nhân dòng khi nối chính sách.')
check('File nguồn không thay đổi', hashlib.sha256(RAW_PATH.read_bytes()).hexdigest() == source_hash, 'So SHA-256.')
source_sums = pd.Series({key: pd.to_numeric(raw[spec['source_column']]).sum() for key, spec in MEASURES.items()})
for name, table in [('Chiều chính', platform_metrics), ('Mức chi tiêu', spend_overall), ('Tháng × nhóm', platform_month)]:
    check(f'Bảo toàn các tổng: {name}', close(table[BASE].sum(), source_sums[BASE]), 'Đối chiếu measure theo cấu hình.')
for key, spec in RATIOS.items():
    expected = safe_divide(overall[spec['numerator']], overall[spec['denominator']])
    check(f'Tỷ lệ tổng đúng công thức: {key}', close(overall[key], expected), 'Tử/mẫu lấy từ metric config.')
roas_spec = RATIOS[ROAS]
check('KPI bộ lọc khớp tính trực tiếp', close(example_metrics[ROAS].iloc[0], safe_divide(
    example_scope[roas_spec['numerator']].sum(), example_scope[roas_spec['denominator']].sum())), example_caption)
for name, scope, table in [('Toàn bộ', df, platform_shares),
                           ('Tập con', selected_platform_scope, selected_platform_shares),
                           ('Matrix', df, cross_shares)]:
    for measure, share in [(REVENUE, 'revenue_contribution'), (SPEND, 'spend_share')]:
        total = scope[measure].sum()
        valid = close(table[share].sum(), 1) if total != 0 else table[share].isna().all()
        check(f'Tỷ trọng {name}: {share}', valid, 'Tổng 100% khi mẫu số khác 0; nếu không thì trống.')
empty = aggregate_metrics(df.iloc[:0])
check('Tập rỗng không tự thành 0', empty[OBS].iloc[0] == 0 and empty[BASE + list(RATIOS)].isna().all().all(), 'KPI trống.')
zero = df.iloc[:1].copy()
zero[roas_spec['denominator']] = 0
check('Mẫu số bằng 0 trả về trống', np.isnan(aggregate_metrics(zero)[ROAS].iloc[0]), 'Không tạo infinity.')
check('Phân loại giữ mọi quan sát', df[LEVEL].notna().all() and spend_counts[OBS].sum() == len(df),
      'Nhóm chưa có chính sách được giữ lại, không xóa.')
sample_threshold = spend_thresholds.iloc[0]
low, high = sample_threshold[LOW], sample_threshold[HIGH]
boundary = classify_spend([low, np.nextafter(low, high), high, np.nextafter(high, np.inf), -1], low, high)
check('Biên ngưỡng đúng chính sách', list(boundary) == [SPEND_ORDER[0], SPEND_ORDER[1], SPEND_ORDER[1],
                                                       SPEND_ORDER[2], UNCLASSIFIED], 'Không gán cứng ngưỡng mẫu.')
check('Thiếu ngưỡng được đánh dấu', classify_spend([low], [np.nan], [np.nan])[0] == UNCLASSIFIED,
      'Không tự đoán chính sách cho nhóm mới.')
missing_level = spend_response(df.loc[df[LEVEL].ne(SPEND_ORDER[1])])
check('Response không nhảy mức', missing_level.loc[missing_level[LEVEL].eq(SPEND_ORDER[-1]),
      list(RESPONSE_COLS.values())].isna().all().all(), 'Không so mức cuối với mức đầu nếu thiếu mức giữa.')
filtered_scope = select_scope(df, filters=example_filters)
filtered_window = compare_window(filtered_scope, window_start, window_end)
previous_start = pd.Timestamp(filtered_window.previous_start.iloc[0])
previous_end = pd.Timestamp(filtered_window.previous_end.iloc[0])
previous_scope = select_scope(filtered_scope, previous_start, previous_end)
expected_previous = aggregate_metrics(previous_scope).iloc[0]
check('Kỳ trước giữ bộ lọc', all(close(row.previous, expected_previous[row.measure])
      for row in filtered_window.itertuples()), 'Giữ tổ hợp ví dụ lấy từ dữ liệu.')
unsupported = compare_window(df, SOURCE_START, SOURCE_START)
check('Kỳ trước ngoài nguồn không có Growth', unsupported.growth.isna().all(), 'Không cố so với kỳ thiếu.')
actual_months = sorted(df[MONTH].unique())
if len(actual_months) >= 3:
    gap_month = pd.Timestamp(actual_months[1])
    after_gap = gap_month.to_period(A['month_frequency']) + 1
    gap_result = calendar_growth(df.loc[df[MONTH].ne(gap_month)], A['month_frequency'])
    selected_gap = gap_result.loc[gap_result.period.eq(after_gap)]
    check('Growth không nhảy kỳ vắng', not selected_gap.empty and selected_gap[
        [f'{key}_growth' for key in A['growth_measures']]].isna().all().all(), 'Tạo khoảng trống từ lịch dữ liệu.')
else:
    check('Growth không nhảy kỳ vắng', True, 'Nguồn chưa đủ kỳ để thử khoảng trống.', skip=True)
check('Độ dài kỳ theo cấu hình', window_comparison.days_each_period.eq(window_days).all()
      and pd.Timestamp(window_comparison.previous_end.iloc[0]) < window_start, 'Không chồng hai kỳ.')
check('Khóa tháng × nhóm duy nhất', not platform_month.duplicated([MONTH, PRIMARY]).any()
      and len(platform_month) == df.groupby([MONTH, PRIMARY]).ngroups,
      'Số nhóm kỳ vọng lấy từ quan sát, không giả định lưới luôn đầy đủ.')
for measure, output in YIELD_COLS.items():
    check(f'Yield đúng tỷ số: {measure}', close(platform_month[output], safe_divide(
        platform_month[measure], platform_month[SPEND])), 'Theo outcome và Spend đã cấu hình.')
expected_budget = baseline_scope[SPEND].sum() / len(baseline_months) if baseline_has_all_months else np.nan
check('Baseline đúng kỳ hiện tại', close(baseline_monthly_budget, expected_budget)
      and close(baseline_allocation[SPEND].sum(), baseline_scope[SPEND].sum()),
      'Không so với ngân sách hoặc tỷ trọng lịch sử viết sẵn.')
check('Histogram không rơi dòng', roas_distribution[OBS].sum() == len(df), 'Kể cả nhóm chưa xác định.')
check('raw giữ nguyên nguồn', raw.equals(pd.read_csv(RAW_PATH, encoding=SEM['source']['encoding'],
      dtype=str, keep_default_na=False)), 'Không giả định số cột nguồn cố định.')
validation_results = pd.DataFrame(checks)
show('Kiểm tra nhất quán', validation_results)
failed = validation_results.loc[validation_results.result.eq('FAIL')]
if not failed.empty:
    raise AssertionError('Kiểm tra thất bại: ' + '; '.join(failed['check']))
display(Markdown(f'**{validation_results.result.eq("PASS").sum()} PASS**, '
                 f'**{validation_results.result.eq("SKIP").sum()} SKIP**, không có FAIL.'))

**Kiểm tra nhất quán**

,check,result,detail
0,Bảo toàn số quan sát và mã dòng,PASS,"Không loại tổ hợp chiều lặp, không nhân dòng khi nối chính sách."
1,File nguồn không thay đổi,PASS,So SHA-256.
2,Bảo toàn các tổng: Chiều chính,PASS,Đối chiếu measure theo cấu hình.
3,Bảo toàn các tổng: Mức chi tiêu,PASS,Đối chiếu measure theo cấu hình.
4,Bảo toàn các tổng: Tháng × nhóm,PASS,Đối chiếu measure theo cấu hình.
5,Tỷ lệ tổng đúng công thức: ctr,PASS,Tử/mẫu lấy từ metric config.
6,Tỷ lệ tổng đúng công thức: cpc,PASS,Tử/mẫu lấy từ metric config.
7,Tỷ lệ tổng đúng công thức: conversion_rate,PASS,Tử/mẫu lấy từ metric config.
8,Tỷ lệ tổng đúng công thức: cpa,PASS,Tử/mẫu lấy từ metric config.
9,Tỷ lệ tổng đúng công thức: roas,PASS,Tử/mẫu lấy từ metric config.


**34 PASS**, **0 SKIP**, không có FAIL.

## 14. Kết luận và cách thay đổi nguồn

Kết quả cụ thể nằm trong các bảng đã chạy; không ghi sẵn số quan sát, nhóm, KPI hoặc ngân sách vào code. Đổi schema nguồn: sửa `source_column` ở `config/semantic.yaml`. Thêm measure/tỷ lệ: bổ sung định nghĩa và tham chiếu hợp lệ tại đó. Đổi quy tắc kỳ, grain, ngưỡng hỗ trợ hoặc histogram: sửa mục `analysis`. Đổi chính sách Country × Industry: cập nhật `config/spend_thresholds.yaml` có nguồn xác nhận.

Các ID ngữ nghĩa và công thức vẫn cần được khai báo vì chương trình không thể tự suy ra ý nghĩa nghiệp vụ của cột. Nếu thêm khái niệm hoàn toàn mới, cần bổ sung định nghĩa thay vì tự động tính mọi cột số.

| Dùng cho báo cáo 5.3 | Bằng chứng |
| --- | --- |
| Định nghĩa measure/KPI | Catalog sinh từ cùng cấu hình với hàm tính |
| Tổng hợp và bộ lọc | KPI tổng, các nhóm thực tế và ví dụ được chọn từ nguồn |
| Kỳ so sánh/tỷ trọng | Kỳ lịch, khoảng ngày, tập con và matrix |
| Scalability | Chính sách có nguồn, Unclassified và Response |
| Growth Pocket | Độ bao phủ theo các grain cấu hình; chưa gắn nhãn cơ hội |
| DSS | Hiệu suất theo tháng × nhóm, Baseline quý cuối nguồn |

Chưa xác nhận tiền tệ, tiêu chí Growth Opportunity, nguồn sinh ngưỡng, sensitivity, Decision Status hoặc tương tác bộ lọc DSS. Chưa kiểm chứng quan hệ bảng vật lý/DAX hoặc tác động nhân quả.

**Xuất file:** điều khiển tại mục `export` trong semantic config, mặc định tắt. Nếu bật, chỉ một bảng đầu vào DSS được ghi vào thư mục con mới trong `data/`; không xuất bảng trung gian hoặc raw. Các YAML là định nghĩa đầu vào, không phải file kết quả xuất thêm.

In [45]:
if EXPORT_PLATFORM_MONTH:
    export_dir = (PROJECT_ROOT / SEM['export']['directory']).resolve()
    data_root = (PROJECT_ROOT / 'data').resolve()
    if data_root not in export_dir.parents:
        raise ValueError('Thư mục xuất phải là thư mục con của data/.')
    filename = SEM['export']['filename']
    if Path(filename).name != filename:
        raise ValueError('Tên file xuất không được chứa đường dẫn.')
    export_dir.mkdir(parents=True, exist_ok=True)
    export_path = export_dir / filename
    platform_month[pm_columns].to_csv(export_path, index=False, encoding=SEM['source']['encoding'],
                                      date_format=SEM['source']['date_format'])
    display(Markdown(f'Đã xuất **{len(platform_month)} dòng** tới `{export_path.relative_to(PROJECT_ROOT)}`.'))
else:
    display(Markdown('**Không xuất CSV hoặc tạo thư mục đầu ra.** Kết quả được lưu trong notebook.'))
display(Markdown(f'Hoàn tất: **{len(df):,} quan sát**, **{len(platform_month)} nhóm tháng × chiều chính**, '
                 f'**{validation_results.result.eq("PASS").sum()} kiểm tra PASS**. '
                 'Đọc kèm trạng thái chính sách, chất lượng nguồn và các giả định nghiệp vụ.'))

**Không xuất CSV hoặc tạo thư mục đầu ra.** Kết quả được lưu trong notebook.

Hoàn tất: **1,800 quan sát**, **36 nhóm tháng × chiều chính**, **34 kiểm tra PASS**. Đọc kèm trạng thái chính sách, chất lượng nguồn và các giả định nghiệp vụ.